# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This notebook simulates parsing synthetic raw EDI lines and Debezium JSON records into structured Bronze output.

In [ ]:
import os
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, expr, lit

warehouse_path = "/tmp/iceberg_warehouse"
os.makedirs(warehouse_path, exist_ok=True)

spark = SparkSession.builder \
    .appName("Day03_Batch_CDC_Ingestion") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Ensure Bronze Tables exist
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.bronze.raw_x12_payload (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.bronze.raw_cdc_payload (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

### 1. Simulate X12 EDI Batch Ingestion

In [ ]:
edi_data = [
    ("ISA*00*          *00*          *ZZ*1234567890     *ZZ*9876543210     *220101*1200*U*00501*000000001*0*T*:~",),
    ("GS*HC*1234567890*9876543210*20220101*1200*1*X*005010X222A1~",),
    ("ST*837*0001*005010X222A1~",)
]

df_edi = spark.createDataFrame(edi_data, ["value"])
df_x12_bronze = df_edi \
    .withColumn("payload_id", expr("uuid()")) \
    .withColumn("source_system_id", lit("simulated_x12_batch")) \
    .withColumn("raw_payload_json", col("value")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "raw_payload_json", "ingestion_timestamp")

df_x12_bronze.write.format("iceberg").mode("append").saveAsTable("local.bronze.raw_x12_payload")
print("X12 EDI Batch ingested.")
spark.sql("SELECT * FROM local.bronze.raw_x12_payload").show(truncate=False)

### 2. Simulate Debezium CDC Ingestion

In [ ]:
cdc_data = [
    (json.dumps({"payload": {"op": "c", "after": {"id": "1", "name": "Alice"}, "before": None}}),),
    (json.dumps({"payload": {"op": "u", "after": {"id": "1", "name": "Alice Smith"}, "before": {"id": "1", "name": "Alice"}}}),),
    (json.dumps({"payload": {"op": "d", "after": None, "before": {"id": "2", "name": "Bob"}}}),)
]

df_cdc_raw = spark.createDataFrame(cdc_data, ["raw_json"])

df_cdc_bronze = df_cdc_raw \
    .withColumn("payload_id", expr("uuid()")) \
    .withColumn("source_system_id", lit("simulated_debezium_cdc")) \
    .withColumn("raw_payload_json", col("raw_json")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "raw_payload_json", "ingestion_timestamp")

df_cdc_bronze.write.format("iceberg").mode("append").saveAsTable("local.bronze.raw_cdc_payload")
print("Debezium CDC Batch ingested.")
spark.sql("SELECT * FROM local.bronze.raw_cdc_payload").show(truncate=False)